In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set the default DPI for inline display and saved files to 300
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
import warnings
from sklearn.model_selection import train_test_split

# pd.set_option("display.max_rows", None)
# # Optional: also show all columns
# pd.set_option("display.max_columns", None)
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings("ignore")

In [2]:
df  =  pd.read_csv("AIDS_Classification_50000.csv")

In [3]:
time_col = "time"
event_col = "infected"

x = df.drop(columns=[time_col, event_col])
t = df[time_col].values
e = df[event_col].values

### Compute horizons at which we evaluate the performance of DSM

In [18]:
horizons = [0.25, 0.5, 0.75]
times = np.quantile(t[e == 1], horizons).tolist()
print(times)

[496.0, 992.0, 1127.0]


In [5]:
cat_cols = [col for col in x.columns if x[col].nunique() < 10]
num_cols = [col for col in x.columns if col not in cat_cols]

In [6]:
x_train, x_temp, t_train, t_temp, e_train, e_temp = train_test_split(
    x, t, e, test_size=0.30, random_state=42, stratify=e
)
x_test, x_val, t_test, t_val, e_test, e_val  = train_test_split(
    x_temp, t_temp, e_temp, test_size=0.30, random_state=42, stratify=e_temp
)

In [8]:

# Remove max values from test/val (often causes issues in evaluation if they exceed train max)
mask_test = t_test < t_train.max()
x_test = x_test[mask_test]
t_test = t_test[mask_test]
e_test = e_test[mask_test]

mask_val = t_val < t_train.max()
x_val = x_val[mask_val]
t_val = t_val[mask_val]
e_val = e_val[mask_val]

In [9]:
t_test.shape, x_test.shape, e_test.shape

((10480,), (10480, 21), (10480,))

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [11]:
# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            cat_cols,
        ),
    ],
    remainder="passthrough",  # Keep any other columns unchanged (optional)
)

In [12]:
# Feature scaling

x_train = preprocessor.fit_transform(x_train)
x_test = preprocessor.transform(x_test)
x_val = preprocessor.transform(x_val)

#### Setting the parameter grid

In [13]:
# from sklearn.model_selection import ParameterGrid
# param_grid = {
#     "k": [3, 4, 5, 6],
#     "distribution": ["LogNormal", "Weibull"],
#     "learning_rate": [1e-3, 1e-4, 1e-5],
#     "layers": [[], [100], [100, 100], [100, 100, 100]],
# }
# params = ParameterGrid(param_grid)

Model Training and Selection

In [14]:
# from auton_survival.models.dsm import DeepSurvivalMachines
# from sksurv.metrics import concordance_index_ipcw

# models = []

# y_train = np.array(
#     [(e_train[i], t_train[i]) for i in range(len(e_train))],
#     dtype=[("e", bool), ("t", float)],
# )
# y_val = np.array(
#     [(e_val[i], t_val[i]) for i in range(len(e_val))],
#     dtype=[("e", bool), ("t", float)],
# )

# for param in params:
#     print(f"\nTesting params: {param}")
#     model = DeepSurvivalMachines(
#         k=param["k"], distribution=param["distribution"], layers=param["layers"]
#     )
#     # The fit method is called to train the model
#     model.fit(
#         x_train, t_train, e_train, iters=100, learning_rate=param["learning_rate"]
#     )
#     out_risk = model.predict_risk(x_val, t=times[1])
#     c_index = concordance_index_ipcw(y_train, y_val, out_risk[:, 0], times[1])[0]
#     print(f"C-index: {c_index:.4f}")
#     models.append([[c_index, model, param]])

# best_model = max(models)
# print("Best model params:", best_model[0][2])
# print("Best model C-index:", best_model[0][0])
# model = best_model[0][1]

Best model params: {'distribution': 'LogNormal', 'k': 4, 'layers': [100], 'learning_rate': 1e-05}
Best model C-index: 0.6794423497236357

Inference

In [16]:
from auton_survival.models.dsm import DeepSurvivalMachines

model = DeepSurvivalMachines(distribution="LogNormal", k=4, layers=[100])
model.fit(x_train, t_train, e_train,val_data=(x_val, t_val, e_val), iters=100, learning_rate=1e-5)

100%|██████████| 100/100 [01:23<00:00,  1.20it/s]


In [17]:
out_risk = model.predict_risk(x_test, times)
out_survival = model.predict_survival(x_test, times)

Evaluation

In [19]:
from sksurv.metrics import integrated_brier_score, cumulative_dynamic_auc, concordance_index_ipcw, brier_score

cis = []
brs = []

et_train = np.array(
    [(e_train[i], t_train[i]) for i in range(len(e_train))],
    dtype=[("e", bool), ("t", float)],
)
et_test = np.array(
    [(e_test[i], t_test[i]) for i in range(len(e_test))],
    dtype=[("e", bool), ("t", float)],
)
et_val = np.array(
    [(e_val[i], t_val[i]) for i in range(len(e_val))], dtype=[("e", bool), ("t", float)]
)

for i, _ in enumerate(times):
    cis.append(concordance_index_ipcw(et_train, et_test, out_risk[:, i], times[i])[0])
brs.append(brier_score(et_train, et_test, out_survival, times)[1])
roc_auc = []
for i, _ in enumerate(times):
    roc_auc.append(
        cumulative_dynamic_auc(et_train, et_test, out_risk[:, i], times[i])[0]
    )
IBS = integrated_brier_score(et_train, et_test, out_survival, times)
for horizon in enumerate(horizons):
    print(f"For {horizon[1]} quantile,")
    print("TD Concordance Index:", cis[horizon[0]])
    print("Brier Score:", brs[0][horizon[0]])
    print("ROC AUC ", roc_auc[horizon[0]][0], "\n")
print("\n\nIntegrated Brier Score:", IBS)

For 0.25 quantile,
TD Concordance Index: 0.6984114215185268
Brier Score: 0.07161862648690144
ROC AUC  0.706693754519431 

For 0.5 quantile,
TD Concordance Index: 0.6837683784935862
Brier Score: 0.15385970068690719
ROC AUC  0.6959677480286999 

For 0.75 quantile,
TD Concordance Index: 0.653428564280219
Brier Score: 0.2118993384488251
ROC AUC  0.6478776695359829 



Integrated Brier Score: 0.12774542041325906


In [20]:
import numpy as np
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc

# Assuming the following variables are available from your previous cells:
# et_train: Structured array of training events/times
# et_test: Structured array of test events/times
# out_risk: Predicted risks at 'times' (Shape: [n_samples, n_times])
# times: List or array of time horizons

# --- 1. Single ROC-AUC (Integrated AUC) ---
# cumulative_dynamic_auc returns (aucs, mean_auc)
# Passing the full 'out_risk' matrix and 'times' calculates AUC at each time and the mean.
aucs, mean_auc = cumulative_dynamic_auc(et_train, et_test, out_risk, times)

print(f"\nTime-dependent AUCs at {times}: {aucs}")
print(f"Single (Integrated) ROC-AUC: {mean_auc}")


# --- 2. Harrell's Concordance Index ---
# Harrell's C (concordance_index_censored) is the standard calculation (without IPCW).
# It typically evaluates a single risk ranking.
# We can calculate it for each horizon:
print("\nHarrell's C-index per horizon:")
for i, t_val in enumerate(times):
    # Using risk predicted for this specific time horizon
    c_index = concordance_index_censored(et_test["e"], et_test["t"], out_risk[:, i])[0]
    print(f"  At time {t_val:.2f}: {c_index:.4f}")

# If you want a single 'Global' Harrell's C-index, you need a single risk score 
# that represents the patient's risk across the study.
# A common approach is to use the risk at the median horizon or the average risk.
global_risk_score = np.mean(out_risk, axis=1) # Average risk over the evaluated horizons
c_index_global = concordance_index_censored(et_test["e"], et_test["t"], global_risk_score)[0]
print(f"\nSingle (Global) Harrell's C-index (using mean risk): {c_index_global:.4f}")


Time-dependent AUCs at [496.0, 992.0, 1127.0]: [0.70669375 0.69596775 0.64787767]
Single (Integrated) ROC-AUC: 0.6778142560135747

Harrell's C-index per horizon:
  At time 496.00: 0.6694
  At time 992.00: 0.6682
  At time 1127.00: 0.6673

Single (Global) Harrell's C-index (using mean risk): 0.6686


In [21]:
import numpy as np
from sksurv.metrics import concordance_index_censored, cumulative_dynamic_auc

# Assuming 'model' is your DeepSurvivalMachines model (or DeepCoxPH)
# and 'x_test' is your feature matrix.

# --- 1. Global Harrell's C-index (Time-Independent) ---
# We calc expected survival time.
# FIX: Ensure time_grid is a LIST, as auton-survival may not handle numpy arrays for 't' correctly in all versions.

# Define a fine grid of times up to the max time in the training data
time_grid = np.linspace(t_train.min(), t_train.max(), 100).tolist() # Converted to list

# Predict survival curves for all test patients [n_samples, n_times]
surv_curves = model.predict_survival(x_test, t=time_grid)

# Integrate (trapz) to get expected survival time for each patient
expected_survival_time = np.trapz(surv_curves, time_grid, axis=1)

# Risk score is inverse of survival time (Conceptually: -ExpectedTime)
risk_score_global = -1 * expected_survival_time

c_index_global = concordance_index_censored(et_test["e"], et_test["t"], risk_score_global)[0]
print(f"Global Harrell's C-index (using Expected Survival Time): {c_index_global:.4f}")


# --- 2. Integrated ROC-AUC (Time-Independent Summary) ---
# Ensure 'times' is also a list just in case
times_list = list(times) if isinstance(times, np.ndarray) else times

out_risk_horizons = model.predict_risk(x_test, t=times_list) # Risk at specific evaluation horizons

aucs, mean_auc = cumulative_dynamic_auc(et_train, et_test, out_risk_horizons, times_list)
print(f"Single (Integrated) ROC-AUC: {mean_auc:.4f}")


Global Harrell's C-index (using Expected Survival Time): 0.6693
Single (Integrated) ROC-AUC: 0.6778
